# 🚀 XGBoost Model Training

This notebook trains the XGBoost model.

**Prerequisites**: Run `01_preprocessing.ipynb` first to generate preprocessed data.

In [2]:
import sys, os
import warnings
warnings.filterwarnings('ignore')
import pickle

# Ensure PROJECT_ROOT is on the path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd

# Project modules
from src.config import *
from src.utils import get_logger

logger = get_logger('XGBoost')
print('✅ Imports successful')

✅ Imports successful


1. Load Preprocessed Data

In [3]:
# Load preprocessed data from file
preprocess_file = os.path.join(MODEL_DIR, 'preprocessed_data.pkl')

if not os.path.exists(preprocess_file):
    raise FileNotFoundError(f'Please run 01_preprocessing.ipynb first to generate {preprocess_file}')

with open(preprocess_file, 'rb') as f:
    prep = pickle.load(f)

X_train = prep['X_train']
X_val = prep['X_val']
X_test = prep['X_test']
y_train = prep['y_train']
y_val = prep['y_val']
y_test = prep['y_test']

print(f'✅ Preprocessed data loaded')
print(f'   Train: {X_train.shape}')
print(f'   Val:   {X_val.shape}')
print(f'   Test:  {X_test.shape}')

✅ Preprocessed data loaded
   Train: (124012, 217)
   Val:   (26575, 217)
   Test:  (26575, 217)


2. Train XGBoost

In [4]:
from src.training import train_xgboost
import importlib
import src.training

# Reload to get latest version
importlib.reload(src.training)
from src.training import train_xgboost

xgb_model, xgb_history = train_xgboost(X_train, y_train, X_val, y_val, use_smote=True)

print('\n✅ XGBoost training complete')

20:51:13 | Training             | INFO    | ==================================================
20:51:13 | Training             | INFO    |   🚀 Training XGBoost
20:51:13 | Training             | INFO    | ==================================================
20:51:18 | Training             | INFO    |   SMOTE Resampling:
20:51:18 | Training             | INFO    |     Before: 119,573 legit, 4,439 fraud (1:26.9)
20:51:18 | Training             | INFO    |     After:  119,573 legit, 59,786 fraud (1:2.0)
20:51:18 | Training             | INFO    |     Synthetic samples: 55,347
20:51:19 | Training             | INFO    | ⏳ Starting: XGBoost training
20:51:19 | Models               | INFO    |   🚀 XGBoost created (n_estimators=500, max_depth=8, scale_pos_weight=26.9)
20:51:55 | Training             | INFO    | ✅ Finished: XGBoost training (35.8s)
20:51:55 | Training             | INFO    |   Val F1-Score: 0.4315
20:51:55 | Training             | INFO    |   Val ROC-AUC:  0.9172
20:51:55 | Train


✅ XGBoost training complete


3. Evaluate on Test Set

3.5 Training Curves & Feature Importance

In [5]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature Importance
feature_importance = xgb_model.feature_importances_
feature_names = prep['feature_names'] if 'feature_names' in prep else [f'Feature {i}' for i in range(len(feature_importance))]

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(15)

axes[0].barh(importance_df['feature'], importance_df['importance'], color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Importance Score', fontweight='bold')
axes[0].set_title('XGBoost - Top 15 Most Important Features', fontweight='bold')
axes[0].invert_yaxis()

# Model Stats
try:
    booster = xgb_model.get_booster()
    n_trees = booster.num_boosted_rounds()
except:
    n_trees = xgb_model.get_params().get('n_estimators', 'Unknown')

stats_text = f"""
📊 XGBoost Model Statistics

🌳 Trees: {n_trees}
📏 Max Depth: {xgb_model.get_params()['max_depth']}
🎯 Learning Rate: {xgb_model.get_params()['learning_rate']}
⚖️ Scale Pos Weight: {xgb_model.get_params()['scale_pos_weight']:.2f}
📈 Features Used: {len(feature_names)}
⭐ Top Feature: {importance_df.iloc[0]['feature']}
"""

axes[1].text(0.5, 0.5, stats_text,
             ha='center', va='center', fontsize=11, family='monospace',
             bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8),
             transform=axes[1].transAxes)
axes[1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, '03_xgb_importance_and_stats.png'), dpi=150, bbox_inches='tight')
plt.show()

print('✅ XGBoost feature importance and model statistics saved')

✅ XGBoost feature importance and model statistics saved


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Predictions
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Metrics
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
}

print('\n' + '='*50)
print('XGBoost - Test Set Performance')
print('='*50)
for metric, value in metrics.items():
    print(f'{metric:15s}: {value:.4f}')
print('='*50)


XGBoost - Test Set Performance
Accuracy       : 0.9326
Precision      : 0.3111
Recall         : 0.7266
F1-Score       : 0.4357
ROC-AUC        : 0.9198


4. Save Model and Results

In [7]:
# Save results (predictions only, not full model)
xgb_results = {
    'y_pred': y_pred,
    'y_pred_proba': y_pred_proba,
    'metrics': metrics,
    'history': xgb_history,
}

results_file = os.path.join(MODEL_DIR, 'xgboost_results.pkl')
with open(results_file, 'wb') as f:
    pickle.dump(xgb_results, f)

print(f'\n✅ Results saved to {results_file}')
print('\n📝 Next: Run 04_train_hgnn.ipynb')


✅ Results saved to c:\Users\Lekshmi Priya\OneDrive\Documents\GitHub\Credit-Card-Fraud-Detection-System\models\xgboost_results.pkl

📝 Next: Run 04_train_hgnn.ipynb


In [8]:
from sklearn.metrics import confusion_matrix

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

# Total fraud cases in test set
total_fraud_cases = tp + fn

# Fraud cases correctly detected
fraud_correctly_detected = tp

# Fraud detection rate
fraud_detection_rate = (fraud_correctly_detected / total_fraud_cases) * 100 if total_fraud_cases > 0 else 0

print('\n' + '='*70)
print('FRAUD DETECTION ANALYSIS - XGBoost MODEL')
print('='*70)
print(f'\n📊 CONFUSION MATRIX BREAKDOWN:')
print(f'   True Negatives (TN):  {tn:6d}  (Correctly identified non-frauds)')
print(f'   False Positives (FP): {fp:6d}  (Non-frauds incorrectly flagged as fraud)')
print(f'   False Negatives (FN): {fn:6d}  (Frauds missed by the model)')
print(f'   True Positives (TP):  {tp:6d}  (Correctly identified frauds)')

print(f'\n🎯 FRAUD DETECTION RESULTS:')
print(f'   Total fraud cases in test set:  {total_fraud_cases}')
print(f'   Fraud cases detected correctly: {fraud_correctly_detected}')
print(f'   Fraud cases MISSED:             {fn}')
print(f'   Fraud detection rate:           {fraud_detection_rate:.2f}%')

print(f'\n⚠️  FALSE POSITIVES (Type I Errors):')
print(f'   Non-fraudulent transactions flagged as fraud: {fp}')
print(f'   False positive rate: {(fp / (tn + fp)) * 100:.2f}%' if (tn + fp) > 0 else '   False positive rate: N/A')

print('\n' + '='*70)


FRAUD DETECTION ANALYSIS - XGBoost MODEL

📊 CONFUSION MATRIX BREAKDOWN:
   True Negatives (TN):   24094  (Correctly identified non-frauds)
   False Positives (FP):   1530  (Non-frauds incorrectly flagged as fraud)
   False Negatives (FN):    260  (Frauds missed by the model)
   True Positives (TP):     691  (Correctly identified frauds)

🎯 FRAUD DETECTION RESULTS:
   Total fraud cases in test set:  951
   Fraud cases detected correctly: 691
   Fraud cases MISSED:             260
   Fraud detection rate:           72.66%

⚠️  FALSE POSITIVES (Type I Errors):
   Non-fraudulent transactions flagged as fraud: 1530
   False positive rate: 5.97%

